# Graph experiments A / B / C / D — combined results

**Compute runs externally**; this notebook loads all four result CSVs and plots them together.

| letter | architecture | graph from | key contrast |
|--------|--------------|-----------|--------------|
| **A** | receptor co-response | y (signed columns) | `nbr_cor` (functional) vs `nbr_esm` (sequence) |
| **B** | collaborative SVD factorization | y (matrix) | `mf_only` = pure-y ceiling; `mf_both` vs raw = additivity; cold-collapse of `u_m` |
| **C** | metapath M-P-M-P | y (co-binding molecule nbrs) | `meta`/`meta_full` vs raw (transductive); cold-collapse in inductive |
| **D** | co-non-response receptor graph | y (negative columns) | `nbr_nor` vs `nbr_cor` vs `nbr_esm` |

Reference ceiling = full boost `[ESM1280‖ChemBERTa384]`: transductive ~0.89, inductive ~0.85 AUROC.
All runs: 5 repeats, stratified inductive split, binary weights, standardised embeddings.

### Run the compute (if not done)
```bash
MAX_PARALLEL=10 bash scripts/modeling/eval/run_receptor_coresponse_graph.sh   # A
MAX_PARALLEL=10 bash scripts/modeling/eval/run_collaborative_factorization.sh  # B
MAX_PARALLEL=10 bash scripts/modeling/eval/run_metapath_cf.sh                  # C
MAX_PARALLEL=10 bash scripts/modeling/eval/run_noresponse_graph.sh             # D
```
Missing CSVs are skipped gracefully — partial runs still render.

In [ ]:
import pathlib, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import t as student_t

ROOT = pathlib.Path.cwd()
while not (ROOT / 'pyproject.toml').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('run from inside the orbind repository')
    ROOT = ROOT.parent

FIGURES_DIR = ROOT / 'notebooks/figures'; FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS = ['AUROC', 'AUPRC', 'MCC', 'F1']
REGIMES = ['transductive', 'inductive_molecule']

def ci95_half(x):
    x = pd.Series(x).dropna().astype(float)
    if len(x) < 2:
        return np.nan
    return float(student_t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x)))

# one spec per architecture: where its CSV lives, and how to order/label/colour its bars
ARCHS = [
    dict(letter='A', name='receptor co-response', dir='results/graph/receptor_coresponse',
         csv='receptor_coresponse_runs.csv', fig='abcd_A.png',
         order=['raw', 'nbr', 'nbr_esm', 'nbr_cor', 'own_esm', 'own_cor'],
         label={'raw': 'raw', 'nbr': 'nbr', 'nbr_esm': 'nbr\n+ESM', 'nbr_cor': 'nbr\n+co-resp',
                'own_esm': 'own\n+ESM', 'own_cor': 'own\n+co-resp'},
         color={'raw': '#9a9a95', 'nbr': '#c8c8c2', 'nbr_esm': '#2a78d6', 'nbr_cor': '#e07b39',
                'own_esm': '#7fa8dd', 'own_cor': '#f0a868'}),
    dict(letter='B', name='collaborative factorization', dir='results/graph/collab_factorization',
         csv='collab_factorization_runs.csv', fig='abcd_B.png',
         order=['raw', 'mf_prot', 'mf_mol', 'mf_both', 'mf_only'],
         label={'raw': 'raw', 'mf_prot': '+v_p', 'mf_mol': '+u_m', 'mf_both': '+u+v', 'mf_only': 'u,v\nonly'},
         color={'raw': '#9a9a95', 'mf_prot': '#8fb0e6', 'mf_mol': '#5a8fd6', 'mf_both': '#2a78d6',
                'mf_only': '#8a5cc0'}),
    dict(letter='C', name='metapath M-P-M-P', dir='results/graph/metapath_cf',
         csv='metapath_cf_runs.csv', fig='abcd_C.png',
         order=['raw', 'meta', 'meta_full'],
         label={'raw': 'raw', 'meta': 'meta', 'meta_full': 'meta\nfull'},
         color={'raw': '#9a9a95', 'meta': '#3d9970', 'meta_full': '#1f7a54'}),
    dict(letter='D', name='co-non-response', dir='results/graph/noresponse',
         csv='noresponse_runs.csv', fig='abcd_D.png',
         order=['raw', 'nbr', 'nbr_esm', 'nbr_cor', 'nbr_nor'],
         label={'raw': 'raw', 'nbr': 'nbr', 'nbr_esm': 'nbr\n+ESM', 'nbr_cor': 'nbr\n+co-resp',
                'nbr_nor': 'nbr\n+non-resp'},
         color={'raw': '#9a9a95', 'nbr': '#c8c8c2', 'nbr_esm': '#2a78d6', 'nbr_cor': '#e07b39',
                'nbr_nor': '#b0439b'}),
]

def load_arch(a):
    rd = ROOT / a['dir']
    combined = rd / 'tables' / a['csv']
    runs = rd / 'tables' / 'runs'
    if combined.exists():
        return pd.read_csv(combined)
    if runs.exists() and glob.glob(str(runs / '*.csv')):
        return pd.concat([pd.read_csv(f) for f in sorted(glob.glob(str(runs / '*.csv')))], ignore_index=True)
    return pd.DataFrame()

RES_ALL = {a['letter']: load_arch(a) for a in ARCHS}
for a in ARCHS:
    R = RES_ALL[a['letter']]
    status = f"{len(R)} rows, {R.repeat.nunique()} repeats/regime" if len(R) else 'NO DATA YET'
    print(f"[{a['letter']}] {a['name']:28s} -> {status}")

In [ ]:
# ---- per-architecture bar grids (regimes x metrics), mean +/- 95% t-CI ----
def plot_arch(a):
    RES = RES_ALL[a['letter']]
    if not len(RES):
        print(f"[{a['letter']}] {a['name']}: no data yet — skipped")
        return
    fig, axes = plt.subplots(len(REGIMES), len(METRICS), figsize=(15, 7), squeeze=False)
    for ri, regime in enumerate(REGIMES):
        d = RES[RES.regime == regime]
        bars = [b for b in a['order'] if b in set(d.bar)]
        for mi, metric in enumerate(METRICS):
            ax = axes[ri, mi]
            means = [d[d.bar == b][metric].mean() for b in bars]
            errs = [ci95_half(d[d.bar == b][metric]) for b in bars]
            x = np.arange(len(bars))
            ax.bar(x, means, color=[a['color'][b] for b in bars], edgecolor='white', linewidth=1.2,
                   yerr=np.nan_to_num(errs), error_kw=dict(ecolor='#33322f', capsize=3, lw=1), zorder=3)
            if means:
                ax.axhline(means[0], color=a['color'][bars[0]], ls='--', lw=1, zorder=1)
            ax.set_ylim(0, 1); ax.grid(alpha=.15, axis='y')
            ax.set_xticks(x); ax.set_xticklabels([a['label'][b] for b in bars], fontsize=8)
            for s in ('top', 'right'): ax.spines[s].set_visible(False)
            if ri == 0: ax.set_title(metric, fontsize=11)
            for xi, v in zip(x, means):
                if np.isfinite(v):
                    ax.text(xi, v + .02, f'{v:.3f}', ha='center', va='bottom', fontsize=8)
        nrep = d.repeat.nunique()
        axes[ri, 0].set_ylabel(f'{regime}\n({nrep} rep)', fontsize=10)
    fig.suptitle(f"[{a['letter']}] {a['name']} — mean +/- 95% t-CI (dashed = raw)", fontsize=12.5, y=1.0)
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / a['fig'], dpi=150, bbox_inches='tight')
    plt.show()

for a in ARCHS:
    plot_arch(a)

In [ ]:
# ---- summary: best non-raw bar vs raw, per architecture x regime (the payoff view) ----
rows = []
for a in ARCHS:
    RES = RES_ALL[a['letter']]
    if not len(RES):
        continue
    for regime in REGIMES:
        d = RES[RES.regime == regime]
        if not len(d) or 'raw' not in set(d.bar):
            continue
        raw = d[d.bar == 'raw']['AUROC'].mean()
        nonraw = [b for b in a['order'] if b in set(d.bar) and b != 'raw']
        if not nonraw:
            continue
        best = max(nonraw, key=lambda b: d[d.bar == b]['AUROC'].mean())
        bm = d[d.bar == best]['AUROC'].mean()
        bci = ci95_half(d[d.bar == best]['AUROC'])
        delta = bm - raw
        sig = 'yes' if (bci == bci and abs(delta) > bci) else 'no'   # delta beyond its own CI half-width
        rows.append({'arch': f"{a['letter']} {a['name']}", 'regime': regime,
                     'raw_AUROC': round(raw, 3), 'best_bar': best, 'best_AUROC': round(bm, 3),
                     'delta_vs_raw': round(delta, 3), 'ci95_half': round(bci, 3) if bci == bci else np.nan,
                     'beats_raw?': sig})

if rows:
    summary = pd.DataFrame(rows)
    display(summary)
    print('\ndelta_vs_raw = best non-raw bar minus raw (AUROC).  beats_raw? = |delta| exceeds its 95% CI half-width.')
    print('Reference ceiling: transductive ~0.89, inductive ~0.85. Nothing is expected to clear raw with CIs.')
else:
    print('No results loaded yet — run the compute, then re-run this notebook.')

## How to read

- Each grid is one architecture (rows = regime, cols = metric); dashed line = that architecture's `raw`.
- The **summary table** is the one-glance verdict: for every architecture × regime it picks the best
  non-`raw` bar and reports `delta_vs_raw` and whether that delta clears its own 95% CI half-width
  (`beats_raw?`). Expect `no` almost everywhere — the whole ledger says nothing beats raw boost.
- Read **AUROC** primarily (prevalence-independent; stratified split matches val/test prevalence).
- Architecture-specific diagnostics (co-response Jaccard, SVD energy / cold-factor fraction, co-binding
  similarity, non-response graph distinctness) live in each architecture's own notebook.

Full ledger of all graph attempts: Obsidian *Графовые эксперименты — сводка*.